# Iris Walkthrough — Setup to Scripts

Objectif: montrer la progression locale/notebook avant le passage en scripts et CI/CD.

Etapes:
1. Setup imports + chemins
2. Prep des donnees Iris
3. Train du modele
4. Evaluation + quality gate
5. Transition vers execution script et pipeline

In [1]:
import os
import sys
import tempfile
import pandas as pd

sys.path.insert(0, '../src')
from prep import main as prep_main
from train import main as train_main
from evaluate import main as evaluate_main

/Users/bmretailleau/projets/mlops-azure-lab/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Setup des dossiers de travail

In [2]:
workdir = tempfile.mkdtemp(prefix='iris-nb-')
data_dir = os.path.join(workdir, 'data')
model_dir = os.path.join(workdir, 'model')
workdir, data_dir, model_dir

('/var/folders/z_/wg5ldty11lg7w2vcqpvk35v80000gp/T/iris-nb-r5z1cmsa',
 '/var/folders/z_/wg5ldty11lg7w2vcqpvk35v80000gp/T/iris-nb-r5z1cmsa/data',
 '/var/folders/z_/wg5ldty11lg7w2vcqpvk35v80000gp/T/iris-nb-r5z1cmsa/model')

## 2) Prep (equivalent `prep.py`)

In [3]:
class PrepArgs:
    def __init__(self, output_dir):
        self.output_dir = output_dir

prep_main(PrepArgs(data_dir))

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
print(train_df.shape, test_df.shape)
train_df.head()

Prep done - train: 120, test: 30
(120, 5) (30, 5)


,sepal_length,sepal_width,petal_length,petal_width,target
0,4.4,2.9,1.4,0.2,0
1,4.9,2.5,4.5,1.7,2
2,6.8,2.8,4.8,1.4,1
3,4.9,3.1,1.5,0.1,0
4,5.5,2.5,4.0,1.3,1


## 3) Train (equivalent `train.py`)

In [4]:
class TrainArgs:
    def __init__(self, data_dir, model_dir, max_iter=200):
        self.data_dir = data_dir
        self.model_dir = model_dir
        self.max_iter = max_iter

train_main(TrainArgs(data_dir, model_dir, max_iter=200))
os.listdir(model_dir)

2026/04/27 11:16:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/27 11:16:29 INFO mlflow.store.db.utils: Updating database tables


Train accuracy: 0.9583 - model saved to /var/folders/z_/wg5ldty11lg7w2vcqpvk35v80000gp/T/iris-nb-r5z1cmsa/model


['model.joblib']

## 4) Evaluate (equivalent `evaluate.py`)

In [5]:
class EvalArgs:
    def __init__(self, data_dir, model_dir, min_accuracy=0.90):
        self.data_dir = data_dir
        self.model_dir = model_dir
        self.min_accuracy = min_accuracy

evaluate_main(EvalArgs(data_dir, model_dir, min_accuracy=0.90))

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

Accuracy: 0.9333 | F1 (weighted): 0.9333 | Threshold: 0.9
PASS: model meets quality threshold


## 5) Transition vers scripts et CI/CD

Apres la validation en notebook, passer aux commandes scripts:

```bash
python mlops/data-science/src/prep.py --output_dir /tmp/iris
python mlops/data-science/src/train.py --data_dir /tmp/iris --model_dir /tmp/model
python mlops/data-science/src/evaluate.py --data_dir /tmp/iris --model_dir /tmp/model
pytest tests/ -v
```

Ensuite, le workflow `ci-train.yml` execute le meme enchainement automatiquement dans GitHub Actions.